# Inference Notebook

Ноутбук формирует `predictions.csv` для тестового набора `docs/daimler_mixtures_test.csv`.
Он использует уже обученные baseline-модели из `baseline/models/` и повторяет ту же сборку признаков, что и при обучении.

In [1]:
from __future__ import annotations

import importlib.util
from pathlib import Path

import pandas as pd
from catboost import CatBoostRegressor


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'docs' / 'daimler_mixtures_test.csv').exists() and (candidate / 'baseline' / 'models').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root with docs/ and baseline/models/.')


PROJECT_ROOT = find_project_root(Path.cwd())
TRAINING_SCRIPT = PROJECT_ROOT / 'baseline' / 'code' / 'train_boosting_baseline.py'

spec = importlib.util.spec_from_file_location('boosting_baseline', TRAINING_SCRIPT)
boosting_baseline = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(boosting_baseline)

DATA_DIR = PROJECT_ROOT / 'docs'
MODEL_DIR = PROJECT_ROOT / 'baseline' / 'models'
OUTPUT_PATH = PROJECT_ROOT / 'predictions.csv'

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'MODEL_DIR = {MODEL_DIR}')
print(f'OUTPUT_PATH = {OUTPUT_PATH}')

PROJECT_ROOT = E:\Projects\Neftecode
MODEL_DIR = E:\Projects\Neftecode\baseline\models
OUTPUT_PATH = E:\Projects\Neftecode\predictions.csv


C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train, test, props = boosting_baseline.load_data(DATA_DIR)
property_table, _ = boosting_baseline.build_property_table(props)
feature_config = boosting_baseline.build_feature_config(train)

train_mixtures = train[boosting_baseline.MIXTURE_COLUMNS + boosting_baseline.SCENARIO_COLUMNS].copy()
test_mixtures = test[boosting_baseline.MIXTURE_COLUMNS + boosting_baseline.SCENARIO_COLUMNS].copy()

X_train = boosting_baseline.build_scenario_features(train_mixtures, property_table, feature_config)
X_test = boosting_baseline.build_scenario_features(test_mixtures, property_table, feature_config)

targets = train.groupby('scenario_id')[boosting_baseline.TARGET_COLUMNS].first().sort_index()
X_train = X_train.loc[targets.index]
X_test = X_test.reindex(columns=X_train.columns)

predictions = pd.DataFrame({'scenario_id': X_test.index})

for target_name in boosting_baseline.TARGET_COLUMNS:
    model_path = MODEL_DIR / f"{boosting_baseline.sanitize_name(target_name)}.cbm"
    model = CatBoostRegressor()
    model.load_model(str(model_path))
    target_pred = model.predict(X_test)
    if target_name.startswith('Delta Kin.'):
        target_pred = boosting_baseline.inverse_sign_log1p(target_pred)
    predictions[target_name] = target_pred

predictions.to_csv(OUTPUT_PATH, index=False)
print(f'Saved predictions to: {OUTPUT_PATH}')
predictions.head()

[baseline] loaded data: train_rows=2230, train_scenarios=167, test_rows=524, test_scenarios=40, property_rows=2641
[baseline] property values parsed: numeric_rows=2226, numeric_properties=74
[baseline] property tables built: measured_component_parties=221, typical_components=91, property_features=74
[baseline] merged property table ready: rows=221, cols=76, fallback_fills=719, avg_numeric_coverage=0.149
[baseline] feature config selected: top_components=25, top_families=9
[baseline] top components: Базовое_масло_1, Детергент_1, Загуститель_1, Антипенная_присадка_1, Базовое_масло_2, Базовое_масло_14, Дисперсант_1, Антиоксидант_1, Депрессорная_присадка_1, Соединение_молибдена_1, Депрессорная_присадка_2, Дисперсант_3, Базовое_масло_3, Базовое_масло_5, Дисперсант_2, Антиоксидант_2, Детергент_2, Антиоксидант_3, Базовое_масло_7, Противоизносная_присадка_1, Противоизносная_присадка_2, Детергент_5, Базовое_масло_8, Базовое_масло_4, Антиоксидант_4
[baseline] top families: Базовое_масло, Детерге

[baseline] scenario features ready: shape=(167, 303), missing_ratio=0.227
[baseline] sample feature columns: n_components, n_unique_components, n_unique_families, share_sum, share_mean, share_std, share_min, share_max, share_entropy, scenario__температура_испытания_astm_d445_daimler_oxidation_test_dot_c, scenario__время_испытания_daimler_oxidation_test_dot_ч, scenario__количество_биотоплива_daimler_oxidation_test_dot_масс
[baseline] building scenario features: rows=524, scenarios=40


[baseline] scenario features ready: shape=(40, 303), missing_ratio=0.237
[baseline] sample feature columns: n_components, n_unique_components, n_unique_families, share_sum, share_mean, share_std, share_min, share_max, share_entropy, scenario__температура_испытания_astm_d445_daimler_oxidation_test_dot_c, scenario__время_испытания_daimler_oxidation_test_dot_ч, scenario__количество_биотоплива_daimler_oxidation_test_dot_масс
Saved predictions to: E:\Projects\Neftecode\predictions.csv


,scenario_id,"Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %","Oxidation EOT | DIN 51453 Daimler Oxidation Test (DOT), A/cm"
0,test_1,-15.712827,35.620340
1,test_10,3.805788,69.811012
2,test_11,-2.694484,74.808616
3,test_12,-11.304856,25.291243
4,test_13,-13.243657,23.252505


In [3]:
expected_ids = test['scenario_id'].drop_duplicates().sort_values().tolist()
actual_ids = predictions['scenario_id'].sort_values().tolist()

assert list(predictions.columns) == ['scenario_id', *boosting_baseline.TARGET_COLUMNS]
assert predictions['scenario_id'].is_unique
assert not predictions[boosting_baseline.TARGET_COLUMNS].isna().any().any()
assert actual_ids == expected_ids

print('Validation passed.')
print(f'Rows: {len(predictions)}')
print(predictions.describe(include='all'))

Validation passed.
Rows: 40
       scenario_id  \
count           40   
unique          40   
top         test_1   
freq             1   
mean           NaN   
std            NaN   
min            NaN   
25%            NaN   
50%            NaN   
75%            NaN   
max            NaN   

        Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %  \
count                                           40.000000                           
unique                                                NaN                           
top                                                   NaN                           
freq                                                  NaN                           
mean                                            46.210685                           
std                                            109.354964                           
min                                            -25.570335                           
25%                        